In [1]:
!pip -q install wordcloud nltk scikit-learn

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import re, string, nltk
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score, roc_curve
import matplotlib.pyplot as plt

nltk.download('stopwords')
nltk.download('wordnet')
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


In [2]:
df = pd.read_csv("spam.csv", encoding='latin-1')

# Keep only label and message columns
df = df[['v1', 'v2']]
df.columns = ['label', 'message']

print(f"Shape: {df.shape}")
print(f"Null values:\n{df.isnull().sum()}")
print(f"Duplicates: {df.duplicated().sum()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nFirst 5 rows:\n{df.head()}")

Shape: (5572, 2)
Null values:
label      0
message    0
dtype: int64
Duplicates: 403

Label distribution:
label
ham     4825
spam     747
Name: count, dtype: int64

First 5 rows:
  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [3]:
stop_words = set(stopwords.words('english'))
lemm = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+|www\S+', ' ', text)
    text = re.sub(r'\S+@\S+', ' ', text)
    text = re.sub(r'\d+', ' ', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = [lemm.lemmatize(word) for word in text.split() if word not in stop_words]
    return ' '.join(tokens)

# Remove duplicates
df = df.drop_duplicates()

# Apply cleaning function
df['clean_message'] = df['message'].apply(clean_text)

# Convert labels to numbers (ham=0, spam=1)
df['label'] = df['label'].map({'ham': 0, 'spam': 1})

print("✅ Data cleaned!")
print(f"Shape after removing duplicates: {df.shape}")
print(f"\nFirst 3 rows:\n{df[['label', 'message', 'clean_message']].head(3)}")

✅ Data cleaned!
Shape after removing duplicates: (5169, 3)

First 3 rows:
   label                                            message  \
0      0  Go until jurong point, crazy.. Available only ...   
1      0                      Ok lar... Joking wif u oni...   
2      1  Free entry in 2 a wkly comp to win FA Cup fina...   

                                       clean_message  
0  go jurong point crazy available bugis n great ...  
1                            ok lar joking wif u oni  
2  free entry wkly comp win fa cup final tkts st ...  


In [4]:
X = df['clean_message']
y = df['label']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(" Data split complete!")
print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set: {X_test.shape[0]} samples")
print(f"\nTrain label distribution:\n{y_train.value_counts()}")
print(f"\nTest label distribution:\n{y_test.value_counts()}")

✅ Data split complete!
Training set: 4135 samples
Testing set: 1034 samples

Train label distribution:
label
0    3613
1     522
Name: count, dtype: int64

Test label distribution:
label
0    903
1    131
Name: count, dtype: int64


In [6]:
nb_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
    ('nb', MultinomialNB())
])

nb_pipeline.fit(X_train, y_train)
nb_pred = nb_pipeline.predict(X_test)
nb_proba = nb_pipeline.predict_proba(X_test)[:, 1]

print("=" * 50)
print("NAIVE BAYES RESULTS")
print("=" * 50)
print(f" Accuracy: {accuracy_score(y_test, nb_pred):.4f}")
print(f" ROC-AUC: {roc_auc_score(y_test, nb_proba):.4f}")
print(f"\n{classification_report(y_test, nb_pred, target_names=['Ham', 'Spam'])}")

NAIVE BAYES RESULTS
 Accuracy: 0.9710
 ROC-AUC: 0.9904

              precision    recall  f1-score   support

         Ham       0.97      1.00      0.98       903
        Spam       0.98      0.79      0.87       131

    accuracy                           0.97      1034
   macro avg       0.98      0.89      0.93      1034
weighted avg       0.97      0.97      0.97      1034



In [7]:
lr_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000, ngram_range=(1, 2))),
    ('lr', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

lr_pipeline.fit(X_train, y_train)
lr_pred = lr_pipeline.predict(X_test)
lr_proba = lr_pipeline.predict_proba(X_test)[:, 1]

print("=" * 50)
print("LOGISTIC REGRESSION RESULTS")
print("=" * 50)
print(f"✅ Accuracy: {accuracy_score(y_test, lr_pred):.4f}")
print(f"✅ ROC-AUC: {roc_auc_score(y_test, lr_proba):.4f}")
print(f"\n{classification_report(y_test, lr_pred, target_names=['Ham', 'Spam'])}")

print("\n" + "=" * 50)
print("MODEL COMPARISON")
print("=" * 50)
print(f"Naive Bayes Accuracy: {accuracy_score(y_test, nb_pred):.4f}")
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, lr_pred):.4f}")

LOGISTIC REGRESSION RESULTS
✅ Accuracy: 0.9749
✅ ROC-AUC: 0.9934

              precision    recall  f1-score   support

         Ham       0.99      0.98      0.99       903
        Spam       0.89      0.92      0.90       131

    accuracy                           0.97      1034
   macro avg       0.94      0.95      0.94      1034
weighted avg       0.98      0.97      0.98      1034


MODEL COMPARISON
Naive Bayes Accuracy: 0.9710
Logistic Regression Accuracy: 0.9749


In [8]:
def predict_sms(text, model, model_name):
    cleaned = clean_text(text)
    pred = model.predict([cleaned])[0]
    conf = model.predict_proba([cleaned])[0][pred]
    result = "SPAM" if pred == 1 else "HAM"
    return result, f"{conf:.2%}"

# Test messages
test_messages = [
    "Congratulations! You have won a free iPhone. Click here now.",
    "Hi, are we meeting tomorrow at 3 PM?",
    "URGENT: Claim your prize NOW! Limited time offer!!!",
    "Hey mom, I'll be home by 6 PM",
    "CALL NOW TO ACTIVATE YOUR ACCOUNT IMMEDIATELY"
]

print("=" * 70)
print("PREDICTION TESTS")
print("=" * 70)

for msg in test_messages:
    print(f"\n📨 Message: {msg}")

    nb_result, nb_conf = predict_sms(msg, nb_pipeline, "Naive Bayes")
    lr_result, lr_conf = predict_sms(msg, lr_pipeline, "Logistic Regression")

    print(f"   🤖 Naive Bayes: {nb_result} (Confidence: {nb_conf})")
    print(f"   🤖 Logistic Regression: {lr_result} (Confidence: {lr_conf})")

PREDICTION TESTS

📨 Message: Congratulations! You have won a free iPhone. Click here now.
   🤖 Naive Bayes: SPAM (Confidence: 72.59%)
   🤖 Logistic Regression: SPAM (Confidence: 71.77%)

📨 Message: Hi, are we meeting tomorrow at 3 PM?
   🤖 Naive Bayes: HAM (Confidence: 97.38%)
   🤖 Logistic Regression: HAM (Confidence: 81.09%)

📨 Message: URGENT: Claim your prize NOW! Limited time offer!!!
   🤖 Naive Bayes: SPAM (Confidence: 94.55%)
   🤖 Logistic Regression: SPAM (Confidence: 92.87%)

📨 Message: Hey mom, I'll be home by 6 PM
   🤖 Naive Bayes: HAM (Confidence: 99.46%)
   🤖 Logistic Regression: HAM (Confidence: 94.95%)

📨 Message: CALL NOW TO ACTIVATE YOUR ACCOUNT IMMEDIATELY
   🤖 Naive Bayes: HAM (Confidence: 50.82%)
   🤖 Logistic Regression: SPAM (Confidence: 58.23%)


In [10]:
import pickle
import json

# Save the best model (Logistic Regression)
with open('sms_spam_detector.pkl', 'wb') as f:
    pickle.dump(lr_pipeline, f)

print("✅ Model saved as 'sms_spam_detector.pkl'")

# Save model info
model_info = {
    'model': 'Logistic Regression',
    'accuracy': float(accuracy_score(y_test, lr_pred)),
    'roc_auc': float(roc_auc_score(y_test, lr_proba)),
    'test_samples': int(len(y_test))
}

with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=4)

print("✅ Model info saved as 'model_info.json'")
print(f"\nModel Performance:")
print(f"  Accuracy: {model_info['accuracy']:.4f}")
print(f"  ROC-AUC: {model_info['roc_auc']:.4f}")

✅ Model saved as 'sms_spam_detector.pkl'
✅ Model info saved as 'model_info.json'

Model Performance:
  Accuracy: 0.9749
  ROC-AUC: 0.9934


In [16]:
results_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': lr_pred,
    'correct': y_test.values == lr_pred
})
results_df.to_csv('predictions.csv', index=False)

print("✅ Predictions saved as 'predictions.csv'")

lr_accuracy = accuracy_score(y_test, lr_pred)
lr_roc_auc = roc_auc_score(y_test, lr_proba)
nb_accuracy = accuracy_score(y_test, nb_pred)
nb_roc_auc = roc_auc_score(y_test, nb_proba)

readme_content = "# SMS Spam Detection\n\n"
readme_content += "## Overview\n"
readme_content += "SMS Spam Detection using Machine Learning (Logistic Regression)\n\n"
readme_content += "## Dataset\n"
readme_content += "- Source: Kaggle - SMS Spam Collection Dataset\n"
readme_content += "- Total Messages: ~5,500\n"
readme_content += "- Classes: Ham (Not Spam) & Spam\n\n"
readme_content += "## Models Used\n"
readme_content += "1. Logistic Regression (Primary)\n"
readme_content += f"   - Accuracy: {lr_accuracy:.2%}\n"
readme_content += f"   - ROC-AUC: {lr_roc_auc:.4f}\n\n"
readme_content += "2. Naive Bayes (Baseline)\n"
readme_content += f"   - Accuracy: {nb_accuracy:.2%}\n"
readme_content += f"   - ROC-AUC: {nb_roc_auc:.4f}\n"

with open('README.md', 'w') as f:
    f.write(readme_content)

print("✅ README.md created.")

✅ Predictions saved as 'predictions.csv'
✅ README.md created.


In [17]:
from google.colab import files

print("\n" + "=" * 60)
print("⬇️  DOWNLOADING FILES TO YOUR COMPUTER")
print("=" * 60)

files.download('sms_spam_detector.pkl')
print("✅ Downloaded: sms_spam_detector.pkl")

files.download('model_info.json')
print("✅ Downloaded: model_info.json")

files.download('predictions.csv')
print("✅ Downloaded: predictions.csv")

files.download('README.md')
print("✅ Downloaded: README.md")

print("\n" + "=" * 60)
print("📁 ALL FILES DOWNLOADED - READY FOR GITHUB!")
print("=" * 60)


⬇️  DOWNLOADING FILES TO YOUR COMPUTER


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: sms_spam_detector.pkl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

✅ Downloaded: predictions.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: README.md

📁 ALL FILES DOWNLOADED - READY FOR GITHUB!


In [19]:
import json
import os

print("Creating missing files...\n")

if not os.path.exists('model_info.json'):
    model_info = {
        'model': 'Logistic Regression',
        'accuracy': float(accuracy_score(y_test, lr_pred)),
        'roc_auc': float(roc_auc_score(y_test, lr_proba))
    }
    with open('model_info.json', 'w') as f:
        json.dump(model_info, f, indent=4)
    print("✅ Created: model_info.json")

if not os.path.exists('predictions.csv'):
    results_df = pd.DataFrame({
        'actual': y_test.values,
        'predicted': lr_pred,
        'correct': y_test.values == lr_pred
    })
    results_df.to_csv('predictions.csv', index=False)
    print("✅ Created: predictions.csv")

if not os.path.exists('README.md'):
    readme = "# SMS Spam Detection\n\nLogistic Regression Model for SMS Spam Detection\n"
    with open('README.md', 'w') as f:
        f.write(readme)
    print("✅ Created: README.md")

print("\nAll files ready!")
print("\nNow downloading...")

from google.colab import files
files.download('model_info.json')
files.download('predictions.csv')
files.download('README.md')
files.download('sms_spam_detector.pkl')

print("\n✅ All files downloaded!")

Creating missing files...


All files ready!

Now downloading...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


✅ All files downloaded!
